## Set up the environment

In [ ]:
!pip install transformers --upgrade
!pip install torch torchaudio

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch,torchaudio

##  Load Model and Processor

In [ ]:
model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

## Download a sample audio from HuggingFace & save it locally

In [ ]:
import requests

url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"

r = requests.get(url)

with open("sample.flac", "wb") as f:
    f.write(r.content)

audio, sampling_rate = torchaudio.load("sample.flac")

## Resample Audio to 16kHz

In [ ]:
#audio, sampling_rate = torchaudio.load("/content/sample_audio.mp3") # Load the audio after manual download

if sampling_rate != 16000:
    resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
    audio = resampler(audio)

## Preprocessing

In [ ]:
inputs = processor(
    audio.squeeze().numpy(),
    sampling_rate=16000,
    return_tensors="pt"
)

## Generate transcription & decode output

In [ ]:
with torch.no_grad():
    predicted_ids = model.generate(inputs["input_features"])

transcription = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

print("Transcription:", transcription)

Transcription:  He hoped there would be stew for dinner, turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick, peppered, flour-fattened sauce.
